In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
cfg = common.load_config(dbutils.widgets.get("env"))
g, s = f"{cfg['catalog']}.gold", f"{cfg['catalog']}.silver"
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

def one(sql):
    return spark.sql(sql).first()[0]

In [0]:
f = f"{g}.fact_bhg_event"
n, keys = spark.sql(f"SELECT count(*), count(DISTINCT event_key) FROM {f}").first()
check("9307 events, one key each", (n, keys) == (9307, 9307), (n, keys))

want = {"consult": 1900, "treatment": 506, "room": 268, "pharmacy": 1644, "discount": 845, "tax": 2168, "payment": 1976}
got = {r[0]: r[1] for r in spark.sql(f"SELECT event_type, count(*) FROM {f} GROUP BY 1").collect()}
check("events per type", got == want, got)

bad = one(f"""WITH e AS (SELECT invoice_id,
                 sum(CASE WHEN event_group IN ('charge', 'adjustment') THEN amount ELSE 0 END) AS billed,
                 sum(CASE WHEN event_group = 'collection' THEN amount ELSE 0 END) AS paid
               FROM {f} WHERE invoice_id IS NOT NULL GROUP BY 1)
              SELECT count(*) FROM {s}.invoices i LEFT JOIN e USING (invoice_id)
              WHERE abs(coalesce(e.billed, 0) - i.net_amt) > 0.01
                 OR (i.status = 'paid' AND abs(coalesce(e.paid, 0) - i.net_amt) > 0.01)""")
check("every invoice reconciles: charges + adjustments = net = payments", bad == 0, bad)

for key, dim in [("date_key", "dim_date"), ("patient_key", "dim_patient"), ("doctor_key", "dim_doctor"),
                 ("hospital_key", "dim_hospital"), ("payer_key", "dim_payer")]:
    orphans = one(f"SELECT count(*) FROM {f} e LEFT ANTI JOIN {g}.{dim} d USING ({key}) WHERE e.{key} IS NOT NULL")
    check(f"every {key} exists in {dim}", orphans == 0, orphans)

check("no event without a date or payer", one(f"SELECT count(*) FROM {f} WHERE date_key IS NULL OR payer_key IS NULL") == 0)
unknown = one(f"SELECT count(*) FROM {f} WHERE doctor_key = -1")
check("events of accepted orphans point to the unknown doctor (5)", unknown == 5, unknown)

print(f"\nt19_gold_fact_bhg: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t19_gold_fact_bhg failed")